# 第 29 章互动实验：采样、引导、掩膜条件、低秩微调和修复

用你在作业里写的六个零件，在两个很小的去噪网络上把采样、无分类器引导、空间条件、低秩微调和修复式生成各做一遍。全部数据由代码生成，不需要下载任何权重；作业还没写完时，对应的单元只打印提示。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "29_扩散模型实战"
ASSIGNMENT = "29-1_采样引导与低秩微调.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


## 准备：噪声表和采样循环

噪声表沿用第 20 章（线性表，1000 步）。`run_sampler` 是采样的外层循环：从 999 到 0 等间隔取若干个步数，每一步调用你写的 `ddim_step`。这个单元不涉及别的作业函数，直接运行。

In [ ]:
import copy, math, time
import torch
import torch.nn.functional as F
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("设备：", device, "｜", torch.cuda.get_device_name(0) if device.type == "cuda" else "没有检测到 CUDA，用 CPU（这里的小模型在 CPU 上一两分钟能跑完）")

def implemented(fn, *args, **kwargs):
    """试着调用一次作业函数，返回它是否已经实现。只有作业里没写完的函数（抛出 TODO）才算没实现；别的错误照常抛出来。"""
    try:
        fn(*args, **kwargs)
        return True
    except NotImplementedError as exc:
        if not str(exc).startswith("TODO"):
            raise
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return False

# 噪声表：和第 20 章作业的 linear_beta_schedule、alpha_bars 算出来的是同一张（线性表，1000 步，下标从 0 开始）
T = 1000
ABAR = torch.cumprod(1 - torch.linspace(1e-4, 0.02, T, dtype=torch.float64), 0)
AB = ABAR.float().to(device)

def timesteps(steps):
    """采样时用到的步数下标：从 T-1 到 0 等间隔取 steps 个。"""
    return [round((T - 1) * (1 - i / (steps - 1))) for i in range(steps)]

def temb(t, dim=32):
    """步数的正弦编码。t 是 0 到 T-1 的整数下标。"""
    f = torch.exp(-math.log(1000.0) * torch.arange(dim // 2, device=t.device) / (dim // 2))
    a = t.float()[:, None] * f[None]
    return torch.cat([a.sin(), a.cos()], 1)

@torch.no_grad()
def run_sampler(eps_fn, x, steps, eta=0.0, g=None, after=None):
    """从噪声 x 出发走 steps 步。eps_fn(x, t) 返回噪声预测，每一步用你写的 ddim_step。
    eta=0 是 DDIM，eta=1 是 DDPM（方差取后验方差）。after(x, abar) 在每一步之后调用，修复时用它替换已知区域。"""
    ts = timesteps(steps)
    for i, t in enumerate(ts):
        a_t = float(ABAR[t])
        a_prev = float(ABAR[ts[i + 1]]) if i + 1 < len(ts) else 1.0       # 最后一步走到干净的图
        noise = torch.randn(x.shape, generator=g).to(x.device) if eta > 0 else None
        x = mod.ddim_step(x, eps_fn(x, t), a_t, a_prev, eta, noise)
        if after is not None:
            x = after(x, a_prev)
    return x

print("abar[0] =", round(float(ABAR[0]), 4), "｜abar[999] =", f"{float(ABAR[-1]):.1e}", "｜20 步采样用到的下标：", timesteps(20)[:4], "...", timesteps(20)[-2:])

## 第一部分：二维的 6 团点

数据是平面上的 6 团点，分成 3 类。网络是一个很小的 MLP，训练时有 15% 的样本把类别换成“没有条件”，为实验 3 的引导做准备。这个单元只定义数据、网络和几个评价用的函数。

In [ ]:
R2D, SIG = 2.0, 0.6
ANG = torch.arange(6) * math.pi / 3
MU = torch.stack([R2D * ANG.cos(), R2D * ANG.sin()], 1)        # 6 个团心，在半径 2 的圆上
CLS = torch.tensor([0, 1, 2, 0, 1, 2])                         # 每个团属于哪一类：相对的两团是同一类
NULL = 3                                                       # “没有条件”用的类别编号

def sample_data(n, g, cls=None):
    """真实数据。cls 给定时每个点来自指定的类别。返回 (点, 类别)。"""
    k = torch.randint(0, 6, (n,), generator=g) if cls is None else cls + 3 * torch.randint(0, 2, (n,), generator=g)
    return MU[k] + SIG * torch.randn(n, 2, generator=g), CLS[k]

DIRS = torch.stack([torch.cos(math.pi * torch.arange(32) / 32), torch.sin(math.pi * torch.arange(32) / 32)], 1)

def sliced_w(a, b):
    """切片 Wasserstein 距离：往 32 个方向投影，各自排序后逐个相减。两组点数要相同。"""
    return float(((a @ DIRS.T).sort(0).values - (b @ DIRS.T).sort(0).values).abs().mean())

def blob_std(x, cls=None):
    """把每个点分给最近的团心（给了 cls 就只在该类的两团里找），各团的标准差取平均。"""
    d = ((x[:, None, :] - MU[None]) ** 2).sum(-1)
    if cls is not None:
        d = d.masked_fill(CLS[None] != cls[:, None], float("inf"))
    k = d.argmin(1)
    return float(torch.stack([x[k == b].std(0).mean() for b in range(6) if int((k == b).sum()) >= 5]).mean())

def bayes_class(x):
    """真实分布下最可能的类别，用来判断生成的点符不符合要求的条件。"""
    lw = -((x[:, None, :] - MU[None]) ** 2).sum(-1) / (2 * SIG ** 2)
    return torch.stack([torch.logsumexp(lw[:, CLS == c], 1) for c in range(3)], 1).argmax(1)

class MLP(nn.Module):
    """二维的去噪网络：输入 x_t、步数的编码、类别的嵌入，输出两个数。"""
    def __init__(self, width=128):
        super().__init__()
        self.cls = nn.Embedding(4, 16)
        self.net = nn.Sequential(nn.Linear(2 + 32 + 16, width), nn.SiLU(), nn.Linear(width, width), nn.SiLU(),
                                 nn.Linear(width, width), nn.SiLU(), nn.Linear(width, 2))
    def forward(self, x, t, c):
        return self.net(torch.cat([x, temb(t), self.cls(c)], 1))

def train_2d(kind="eps", steps=1500, p_drop=0.15, seed=0):
    """kind 是网络预测的量：eps（噪声）、x0（原图）或 v。训练时每个样本的类别有 p_drop 的概率换成“没有条件”。"""
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(100 + seed)
    net = MLP().to(device)
    opt = torch.optim.Adam(net.parameters(), 2e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for _ in range(steps):
        x0, c = sample_data(512, g)
        t = torch.randint(0, T, (512,), generator=g)
        eps = torch.randn(512, 2, generator=g)
        c = torch.where(torch.rand(512, generator=g) < p_drop, torch.full_like(c, NULL), c)
        x0, c, t, eps = x0.to(device), c.to(device), t.to(device), eps.to(device)
        a = AB[t][:, None]
        xt = a.sqrt() * x0 + (1 - a).sqrt() * eps                          # 第 20 章的闭式加噪
        target = {"eps": eps, "x0": x0, "v": a.sqrt() * eps - (1 - a).sqrt() * x0}[kind]
        loss = ((net(xt, t, c) - target) ** 2).mean()
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    return net.eval()

def sample_2d(net, n, steps, eta=0.0, cls=None, w=1.0, kind="eps", seed=1):
    """从噪声出发采 n 个点。网络预测的不是噪声时，先用你写的 convert_prediction 换算；给了 cls 时用你写的 cfg_combine 做引导。"""
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, 2, generator=g).to(device)
    null = torch.full((n,), NULL, device=device)
    c = None if cls is None else cls.to(device)
    def predict(x, tt, cond, a):
        out = net(x, tt, cond)
        return out if kind == "eps" else mod.convert_prediction(x, out, a, kind)[1]
    def eps_fn(x, t):
        tt, a = torch.full((n,), t, device=device), float(ABAR[t])
        eu = predict(x, tt, null, a)
        return eu if c is None else mod.cfg_combine(eu, predict(x, tt, c, a), w)
    return run_sampler(eps_fn, x, steps, eta, g).cpu()

g = torch.Generator().manual_seed(8)
pts, lab = sample_data(1500, g)
plt.figure(figsize=(3.6, 3.6))
for c in range(3):
    plt.scatter(pts[lab == c, 0], pts[lab == c, 1], s=3, label=f"类别 {c}")
plt.legend(markerscale=3, fontsize=8); plt.gca().set_aspect("equal"); plt.title("真实数据：6 团，3 类"); plt.show()

## 实验 1：采样器和步数

训练一个预测噪声的网络，再用你写的 `ddim_step` 采样：`eta=0` 是 DDIM，`eta=1` 是 DDPM。步数从 200 减到 5。

先预测：步数减到 5 时，团的位置还对不对？团是变大还是变小？DDIM 和 DDPM 谁离真实分布更近？再运行，对照教程图 2 和 2.3 节的表。

In [ ]:
ddim_ready = implemented(mod.ddim_step, torch.zeros(2, 2), torch.zeros(2, 2), 0.5, 0.9)
net_eps = None
if ddim_ready:
    t0 = time.perf_counter()
    net_eps = train_2d("eps")
    print(f"训练预测噪声的网络用时 {time.perf_counter() - t0:.1f} 秒")
    g = torch.Generator().manual_seed(7)
    real = sample_data(2000, g)[0]
    ref_std = blob_std(real)
    print("两批真实样本之间的距离：", round(sliced_w(sample_data(2000, g)[0], real), 3), "（采样结果到这个水平就分不出好坏了）")
    print("步数    DDIM 的距离  团内标准差比    DDPM 的距离  团内标准差比")
    shown = {}
    for steps in (200, 50, 20, 10, 5):
        row = []
        for eta in (0.0, 1.0):
            x = sample_2d(net_eps, 2000, steps, eta)
            row += [sliced_w(x, real), blob_std(x) / ref_std]
            shown[(steps, eta)] = x
        print(f"{steps:4d}      {row[0]:.3f}        {row[1]:.2f}           {row[2]:.3f}        {row[3]:.2f}")
    fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
    for a, (title, x) in zip(ax, (("真实数据", real), ("DDIM，50 步", shown[(50, 0.0)]), ("DDIM，5 步", shown[(5, 0.0)]), ("DDPM，5 步", shown[(5, 1.0)]))):
        a.scatter(real[:, 0], real[:, 1], s=2, color="lightgray"); a.scatter(x[:, 0], x[:, 1], s=2)
        a.set_xlim(-4.5, 4.5); a.set_ylim(-4.5, 4.5); a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([]); a.set_title(title)
    plt.tight_layout(); plt.show()

## 实验 2：预测噪声、预测原图、预测 v

三个网络结构相同，只是预测的量不同。用你写的 `convert_prediction` 把它们的输出都换算成对原图和对噪声的估计，在噪声最少（t=0）和最多（t=999）的两端各量一次误差。

先预测：预测噪声的网络在哪一端换算成原图时误差最大？预测原图的网络呢？对照教程图 3。

In [ ]:
conv_ready = implemented(mod.convert_prediction, torch.zeros(2, 2), torch.zeros(2, 2), 0.5, "eps")
nets = {}
if conv_ready:
    nets = {"eps": net_eps if net_eps is not None else train_2d("eps"), "x0": train_2d("x0"), "v": train_2d("v")}
    names = {"eps": "预测噪声", "x0": "预测原图", "v": "预测 v"}
    g = torch.Generator().manual_seed(9)
    x0 = sample_data(5000, g)[0].to(device)
    null = torch.full((5000,), NULL, device=device)
    print("把网络的输出换算之后的均方误差       t=0 时对噪声     t=999 时对原图")
    for kind, net in nets.items():
        row = []
        for t, which in ((0, 1), (999, 0)):                    # convert_prediction 返回 (x0, eps, v)
            a = float(ABAR[t])
            eps = torch.randn(5000, 2, generator=g).to(device)
            xt = math.sqrt(a) * x0 + math.sqrt(1 - a) * eps
            with torch.no_grad():
                out = mod.convert_prediction(xt, net(xt, torch.full((5000,), t, device=device), null), a, kind)
            row.append(float(((out[which] - (x0, eps)[which]) ** 2).mean()))
        print(f"{names[kind]:8s}                              {row[0]:8.3f}         {row[1]:8.3f}")
    print("对照：噪声的方差是 1；原图的方差是", round(float(x0.var(0).mean()), 2), "（什么都不知道、只猜平均值时的误差）")
    if ddim_ready:
        g = torch.Generator().manual_seed(7)
        real = sample_data(2000, g)[0]
        print("DDIM 只走很少几步时到真实分布的距离：")
        for kind, net in nets.items():
            print(f"  {names[kind]:8s}", "  ".join(f"{s} 步 {sliced_w(sample_2d(net, 2000, s, kind=kind), real):.2f}" for s in (2, 5, 20)))

## 实验 3：无分类器引导

同一个预测噪声的网络，每一步算两次：一次不给条件，一次给类别，用你写的 `cfg_combine` 组合。引导强度取 0、1、3、8。

先预测：w 从 1 加到 3，条件符合率和团的大小各怎么变？加到 8 呢？对照教程图 4，也可以在教程第 9 节的演示页里拖一拖。

In [ ]:
cfg_ready = ddim_ready and implemented(mod.cfg_combine, torch.zeros(2, 2), torch.zeros(2, 2), 1.0)
if cfg_ready:
    n = 1500
    cls = torch.arange(3).repeat_interleave(n // 3)             # 三个类别各要 500 个
    g = torch.Generator().manual_seed(9)
    real_c = sample_data(n, g, cls)[0]
    ref_std = blob_std(real_c, cls)
    print(f"真实的条件样本：条件符合率 {float((bayes_class(real_c) == cls).float().mean()):.3f}（团之间有重叠，所以到不了 1）")
    fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
    for a, w in zip(ax, (0.0, 1.0, 3.0, 8.0)):
        x = sample_2d(net_eps, n, 50, cls=cls, w=w)
        print(f"w = {w:3.1f}：条件符合率 {float((bayes_class(x) == cls).float().mean()):.3f}，团内标准差比 {blob_std(x, cls) / ref_std:.2f}")
        a.scatter(real_c[:, 0], real_c[:, 1], s=2, color="lightgray"); a.scatter(x[cls == 0, 0], x[cls == 0, 1], s=3)
        a.set_xlim(-5.2, 5.2); a.set_ylim(-5.2, 5.2); a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([]); a.set_title(f"要求类别 0，w = {w:g}")
    plt.tight_layout(); plt.show()

## 第二部分：16×16 的纹理加缺陷

图是竖条纹，缺陷有两种：斑点（老的缺陷，样本很多）和横向划痕（新出现的缺陷，样本很少）。去噪网络是一个 4 万参数的小 UNet。它的每个卷积都可以挂一个低秩增量，挂上之后的前向调用你写的 `lora_forward`，合并调用 `lora_merge`。这个单元只定义数据和模型，直接运行。

In [ ]:
SIZE, PIX_NOISE, PB = 16, 0.02, 8
YY, XX = np.mgrid[0:SIZE, 0:SIZE].astype(np.float64)

def texture(n, rng):
    return 0.5 + 0.15 * np.cos(2 * np.pi * (XX[None] + rng.uniform(0, 4, size=(n, 1, 1))) / 4.0)

def _uv(n, rng, lo, hi, th):
    cy, cx = rng.uniform(lo, hi, size=(2, n, 1, 1))
    return (YY[None] - cy) * np.sin(th) + (XX[None] - cx) * np.cos(th), -(YY[None] - cy) * np.cos(th) + (XX[None] - cx) * np.sin(th)

def make(n, rng, kind):
    """返回 (图, 缺陷掩膜)，都是 (n, 1, 16, 16)。kind：good 良品、blob 斑点、scratch 横向划痕。图已经减 0.5 再除以 0.2。"""
    clean = texture(n, rng)
    prof = np.zeros_like(clean)
    if kind == "blob":
        s, ratio, th = rng.uniform(1.0, 1.6, size=(n, 1, 1)), rng.uniform(1.0, 1.8, size=(n, 1, 1)), rng.uniform(0, np.pi, size=(n, 1, 1))
        u, v = _uv(n, rng, 3.5, SIZE - 4.5, th)
        prof = np.exp(-(u ** 2 / (2 * (s * ratio) ** 2) + v ** 2 / (2 * s ** 2)))
    elif kind == "scratch":
        length, th = rng.uniform(5, 9, size=(n, 1, 1)), np.deg2rad(rng.uniform(-20, 20, size=(n, 1, 1)))
        u, v = _uv(n, rng, 4.5, SIZE - 5.5, th)
        prof = np.exp(-np.hypot(np.maximum(np.abs(u) - length / 2, 0), v) ** 2 / (2 * 0.7 ** 2))
    if kind != "good":
        clean = clean + rng.choice([-1.0, 1.0], size=(n, 1, 1)) * rng.uniform(0.2, 0.4, size=(n, 1, 1)) * prof
    img = clean * rng.uniform(0.9, 1.1, size=(n, 1, 1)) + rng.normal(0, PIX_NOISE, size=clean.shape)
    return torch.tensor((img - 0.5) / 0.2, dtype=torch.float32)[:, None], torch.tensor(prof > 0.3, dtype=torch.float32)[:, None]

def residual(x):
    """条纹沿竖直方向不变，减掉每一列的中位数，剩下的就是缺陷和噪声。"""
    return x - x.median(dim=2, keepdim=True).values

class LConv(nn.Module):
    """3x3 卷积，可以挂一个低秩增量。挂上之后的前向把输入展开成 3x3 邻域，用你写的 lora_forward 计算（卷积就是对邻域做线性层）。"""
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 3, stride, 1)
        self.A = self.B = None
        self.alpha = 1.0
    def add_lora(self, r, g):
        w = self.conv.weight
        cout, fan = w.shape[0], w[0].numel()
        r = min(r, cout, fan)                                   # 秩不可能超过权重矩阵较短的那一边
        self.A = nn.Parameter((torch.randn(r, fan, generator=g) / math.sqrt(fan)).to(w.device))
        self.B = nn.Parameter(torch.zeros(cout, r, device=w.device))     # B 从 0 开始：微调开始时不改变原模型
        self.alpha = float(r)
    def merged_weight(self):
        w = self.conv.weight
        return mod.lora_merge(w.flatten(1), self.A, self.B, self.alpha).view_as(w)
    def forward(self, x):
        if self.A is None:
            return self.conv(x)
        b, _, h, w = x.shape
        st = self.conv.stride[0]
        y = mod.lora_forward(F.unfold(x, 3, padding=1, stride=st).transpose(1, 2), self.conv.weight.flatten(1), self.conv.bias, self.A, self.B, self.alpha)
        return y.transpose(1, 2).reshape(b, -1, h // st, w // st)

class UNet(nn.Module):
    """很小的去噪网络：16x16 和 8x8 两个尺度，8 个 3x3 卷积，预测噪声。"""
    def __init__(self, c=16):
        super().__init__()
        self.c = c
        self.t = nn.Sequential(nn.Linear(32, 64), nn.SiLU(), nn.Linear(64, 64))
        self.c1, self.c2, self.d, self.m1, self.m2 = LConv(1, c), LConv(c, c), LConv(c, 2 * c, 2), LConv(2 * c, 2 * c), LConv(2 * c, 2 * c)
        self.u, self.o1, self.o2 = LConv(2 * c, c), LConv(c, c), LConv(c, 1)
        self.f1, self.f2 = nn.Linear(64, c), nn.Linear(64, 2 * c)
    def convs(self):
        return [self.c1, self.c2, self.d, self.m1, self.m2, self.u, self.o1, self.o2]
    def encode(self, x, e, hint=None):
        h = F.silu(self.c1(x))
        if hint is not None:
            h = h + hint
        h1 = F.silu(self.c2(h + self.f1(e)[:, :, None, None]))
        return h1, F.silu(self.m1(F.silu(self.d(h1)) + self.f2(e)[:, :, None, None]))
    def forward(self, x, t, ctrl=None):
        h1, h2 = self.encode(x, self.t(temb(t)))
        if ctrl is not None:                                    # 控制旁路给出的两个残差，加在两个尺度的特征上
            h1, h2 = h1 + ctrl[0], h2 + ctrl[1]
        u = F.interpolate(F.silu(self.u(F.silu(self.m2(h2)))), scale_factor=2, mode="nearest")
        return self.o2(F.silu(self.o1(u + h1)))

class Control(nn.Module):
    """ControlNet 式的旁路：原模型编码器的一份可训练拷贝，输入处加上掩膜的特征；输出经过两个 1x1 卷积加回原模型。
    zero_init=True 时这两个 1x1 卷积从 0 开始。"""
    def __init__(self, base, zero_init=True):
        super().__init__()
        self.enc = copy.deepcopy(base)
        for name in ("m2", "u", "o1", "o2"):
            delattr(self.enc, name)
        for p in self.enc.parameters():
            p.requires_grad_(True)
        self.hint = nn.Sequential(nn.Conv2d(1, base.c, 3, padding=1), nn.SiLU(), nn.Conv2d(base.c, base.c, 3, padding=1))
        self.z1, self.z2 = nn.Conv2d(base.c, base.c, 1), nn.Conv2d(2 * base.c, 2 * base.c, 1)
        if zero_init:
            for z in (self.z1, self.z2):
                nn.init.zeros_(z.weight); nn.init.zeros_(z.bias)
    def forward(self, x, t, mask):
        h1, h2 = self.enc.encode(x, self.enc.t(temb(t)), self.hint(mask))
        return self.z1(h1), self.z2(h2)

def train_diffusion(net, data, steps, batch, lr, seed, params=None, ctrl=None, masks=None):
    """噪声预测损失。params 为 None 时训练 net 的全部参数；给了 ctrl 时把掩膜当条件，只训练 params。返回每一步的损失。"""
    g = torch.Generator().manual_seed(1000 + seed)
    opt = torch.optim.Adam(list(net.parameters()) if params is None else params, lr)
    log = []
    for _ in range(steps):
        idx = torch.randint(0, len(data), (batch,), generator=g)
        x0 = data[idx].to(device)
        t = torch.randint(0, T, (batch,), generator=g).to(device)
        eps = torch.randn(x0.shape, generator=g).to(device)
        a = AB[t][:, None, None, None]
        xt = a.sqrt() * x0 + (1 - a).sqrt() * eps
        c = ctrl(xt, t, masks[idx].to(device)) if ctrl is not None else None
        loss = ((net(xt, t, c) - eps) ** 2).mean()
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        log.append(float(loss.detach()))
    return log

def sample_img(net, n, seed, steps=20, ctrl=None, mask=None, after=None):
    """DDIM 采样 n 张图，返回在 CPU 上的结果。after(x, abar, g) 在每一步之后调用。"""
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, 1, SIZE, SIZE, generator=g).to(device)
    m = None if mask is None else mask.to(device)
    def eps_fn(x, t):
        tt = torch.full((n,), t, device=device)
        return net(x, tt, ctrl(x, tt, m) if ctrl is not None else None)
    return run_sampler(eps_fn, x, steps, 0.0, g, None if after is None else (lambda x, a: after(x, a, g))).cpu()

class Clf(nn.Module):
    """三层卷积加全局最大池化的小分类器，用来判断一张图像良品、斑点还是横向划痕。"""
    def __init__(self, out=3):
        super().__init__()
        self.c1, self.c2, self.c3 = nn.Conv2d(1, 12, 3, padding=1), nn.Conv2d(12, 24, 3, stride=2, padding=1), nn.Conv2d(24, 24, 3, padding=1)
        self.fc = nn.Linear(24, out)
    def forward(self, x):
        return self.fc(F.relu(self.c3(F.relu(self.c2(F.relu(self.c1(x)))))).amax(dim=(2, 3)))

def nn_dist(a, b):
    """a 里每张图到 b 里最近的一张的距离（逐像素差的均方根）。"""
    return (torch.cdist(a.flatten(1), b.flatten(1)) / math.sqrt(a[0].numel())).min(1).values

def show_row(ax, imgs, masks=None, title=None, color="c"):
    for i, a in enumerate(ax):
        a.imshow(imgs[i, 0], cmap="gray", vmin=-2.2, vmax=2.2); a.set_xticks([]); a.set_yticks([])
        if masks is not None and float(masks[i].max()) > 0:
            a.contour(masks[i, 0].numpy(), levels=[0.5], colors=color, linewidths=1)
    if title:
        ax[0].set_ylabel(title, fontsize=9)

rng = np.random.default_rng(0)
blob, blob_m = make(3000, rng, "blob")          # 老的缺陷：斑点，很多
good, _ = make(1000, rng, "good")
scr, _ = make(1700, rng, "scratch")             # 新的缺陷：横向划痕。前 1000 张只用来训练判别器，微调时只拿其中十几张
fig, ax = plt.subplots(3, 8, figsize=(10, 4))
show_row(ax[0], good, title="良品"); show_row(ax[1], blob, blob_m, "斑点"); show_row(ax[2], scr, title="横向划痕")
plt.tight_layout(); plt.show()

## 实验 4：用掩膜控制缺陷的位置

先在斑点图上训练基础模型（1200 步，CPU 上二三十秒），再冻结它，只训练一个读入掩膜的旁路。旁路最后的两个 1×1 卷积从 0 开始。

先预测：训练之前，接上旁路会让基础模型的输出改变多少？训练之后，不加条件和加条件的样本，缺陷落在指定掩膜里的比例各是多少？对照教程图 5。

In [ ]:
base = ctrl = judge = None
if ddim_ready:
    # 判别器：只是评价工具，和作业无关
    torch.manual_seed(0)
    g = torch.Generator().manual_seed(77)
    xs, ys = torch.cat([good[:800], blob[:800], scr[:800]]), torch.arange(3).repeat_interleave(800)
    judge = Clf().to(device)
    opt = torch.optim.Adam(judge.parameters(), 3e-3)
    for _ in range(500):
        idx = torch.randint(0, len(xs), (32,), generator=g)
        loss = F.cross_entropy(judge(xs[idx].to(device)), ys[idx].to(device))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    judge.eval()

    def rates(x):
        """判别器把一批图判成 良品 / 斑点 / 横向划痕 的比例。"""
        with torch.no_grad():
            p = judge(x.to(device)).argmax(1).cpu()
        return [round(float((p == c).float().mean()), 2) for c in range(3)]

    THR = float(torch.quantile(residual(good).abs().amax(dim=(1, 2, 3)), 0.99))      # 良品图残差最大值的 99% 分位

    def mask_metrics(x, m):
        """生成的缺陷和指定掩膜对得上的程度：掩膜内能量占比、掩膜内有缺陷的比例。"""
        r = residual(x)
        e = (r * (r.abs() > THR)) ** 2
        tot, ins = e.sum((1, 2, 3)), (e * m).sum((1, 2, 3))
        has = tot > 0
        return round(float((ins[has] / tot[has]).mean()), 2), round(float(((r.abs() * m).amax(dim=(1, 2, 3)) > THR).float().mean()), 2)

    t0 = time.perf_counter()
    torch.manual_seed(0)
    base = UNet().to(device)
    log = train_diffusion(base, blob, 1200, 32, 2e-3, seed=0)                         # 基础模型只见过斑点
    for p in base.parameters():
        p.requires_grad_(False)
    print(f"基础模型训练 {time.perf_counter() - t0:.0f} 秒，最后 100 步的平均损失 {np.mean(log[-100:]):.3f}")
    req_x, req_m = make(200, np.random.default_rng(5), "blob")                       # 200 个“指定的掩膜”
    x_free = sample_img(base, 200, seed=3)
    print("真实斑点图被判成 良品/斑点/划痕：", rates(blob[2000:2200]), "｜基础模型的样本：", rates(x_free))

    torch.manual_seed(1)
    ctrl = Control(base).to(device)
    g = torch.Generator().manual_seed(5)
    xt, tt = torch.randn(64, 1, SIZE, SIZE, generator=g).to(device), torch.randint(0, T, (64,), generator=g).to(device)
    with torch.no_grad():
        change = float((base(xt, tt, ctrl(xt, tt, req_m[:64].to(device))) - base(xt, tt)).abs().max())
    print("训练之前，接上旁路让输出改变的最大值：", change)
    t0 = time.perf_counter()
    train_diffusion(base, blob, 300, 32, 2e-3, seed=1, params=list(ctrl.parameters()), ctrl=ctrl, masks=blob_m)
    print(f"旁路训练 {time.perf_counter() - t0:.0f} 秒（基础模型冻结，只训练旁路）")
    x_ctrl = sample_img(base, 200, seed=3, ctrl=ctrl, mask=req_m)
    print("（掩膜内能量占比，掩膜内有缺陷的比例）  不加条件：", mask_metrics(x_free, req_m), "｜加掩膜条件：", mask_metrics(x_ctrl, req_m), "｜真实图对自己的掩膜：", mask_metrics(req_x, req_m))
    fig, ax = plt.subplots(2, 8, figsize=(10, 2.8))
    show_row(ax[0], x_free, req_m, "不加条件", "r"); show_row(ax[1], x_ctrl, req_m, "加掩膜条件", "r")
    plt.tight_layout(); plt.show()

## 实验 5：低秩微调，学一类新缺陷

基础模型只见过斑点。现在分别给它 16 张和 4 张横向划痕图，冻结原有权重，只训练每个卷积上秩 4 的增量，再把增量并回权重。

先预测：合并前后网络的输出差多少？只有 4 张图、训练 800 步时，生成的样本和那 4 张训练图会有多像？对照教程图 6。

In [ ]:
lora_ready = (base is not None
              and implemented(mod.lora_forward, torch.zeros(1, 3), torch.zeros(2, 3), None, torch.zeros(1, 3), torch.zeros(2, 1), 1.0)
              and implemented(mod.lora_merge, torch.zeros(2, 3), torch.zeros(1, 3), torch.zeros(2, 1), 1.0))
if lora_ready:
    def finetune(train, r, steps, seed=0):
        """复制一份基础模型，给 8 个卷积各挂一个秩 r 的增量，只训练增量。返回 (带旁路的网络, 可训练参数个数)。"""
        net = copy.deepcopy(base)
        g = torch.Generator().manual_seed(40 + seed)
        for c in net.convs():
            c.add_lora(r, g)
        params = [p for c in net.convs() for p in (c.A, c.B)]
        train_diffusion(net, train, steps, 16, 5e-3, seed=seed, params=params)
        return net.eval(), sum(p.numel() for p in params)

    def merged_copy(net):
        """用你写的 lora_merge 把增量并回卷积权重，得到一个不带旁路的普通网络。"""
        out = copy.deepcopy(net)
        with torch.no_grad():
            for c in out.convs():
                c.conv.weight.copy_(c.merged_weight())
                c.A = c.B = None
        return out.eval()

    held = scr[1200:1700]                                         # 留出的真实划痕，用来给“离训练集多近算正常”定一个参照
    print("微调之前，基础模型的样本被判成 良品/斑点/划痕：", rates(sample_img(base, 200, seed=5)))
    shown = {}
    for K, steps in ((16, 300), (4, 800)):
        train = scr[1000:1000 + K]
        net, n_par = finetune(train, 4, steps)
        merged = merged_copy(net)
        g = torch.Generator().manual_seed(6)
        xt, tt = torch.randn(64, 1, SIZE, SIZE, generator=g).to(device), torch.randint(0, T, (64,), generator=g).to(device)
        with torch.no_grad():
            gap, moved = float((merged(xt, tt) - net(xt, tt)).abs().max()), float((merged(xt, tt) - base(xt, tt)).abs().max())
        x = sample_img(merged, 200, seed=5)
        d, dr = nn_dist(x, train), nn_dist(held, train)
        print(f"{K} 张真实划痕，秩 4，微调 {steps} 步（可训练参数 {n_par} 个）：合并前后输出最大差 {gap:.1e}，和基础模型的最大差 {moved:.2f}")
        print(f"    样本被判成 良品/斑点/划痕 {rates(x)}；到训练集最近邻距离的中位数 {float(d.median()):.3f}（留出的真实划痕 {float(dr.median()):.3f}）；"
              f"复制比例 {float((d < torch.quantile(dr, 0.01)).float().mean()):.3f}（真实新样本约 0.01）")
        shown[K] = (train, x[d.argsort()[:8]])
    fig, ax = plt.subplots(3, 8, figsize=(10, 4))
    show_row(ax[0], shown[16][0], title="16 张里的 8 张训练图"); show_row(ax[1], shown[16][1], title="16 张：离训练集最近的 8 个样本")
    show_row(ax[2], torch.cat([shown[4][0], shown[4][1][:4]]), title="4 张训练图，和最像它们的 4 个样本")
    plt.tight_layout(); plt.show()

## 实验 6：在良品图上长出一个缺陷

给一张良品图和一个方框，只在方框里重新生成，位置和形状由掩膜条件指定。每一步采样之后，用你写的 `inpaint_replace` 把方框外换成良品图加噪到当前噪声水平的结果。对照组是“采样时完全不管良品图，最后才把方框外贴回去”。

先预测：两种做法的方框外各被改动了多少？方框的边界上哪一种更容易看出接缝？对照教程图 7 和 6.2 节。

In [ ]:
inpaint_ready = ctrl is not None and implemented(mod.inpaint_replace, torch.zeros(1, 1, 2, 2), torch.zeros(1, 1, 2, 2), torch.ones(1, 1, 2, 2), 0.5, torch.zeros(1, 1, 2, 2))
if inpaint_ready:
    n = 200
    tgt, m = good[:n], req_m[:n]                                  # 200 张良品图，每张指定一个斑点形状的掩膜
    ys, xs = torch.tensor(YY, dtype=torch.float32), torch.tensor(XX, dtype=torch.float32)
    cy = ((m[:, 0] * ys).sum((1, 2)) / m.sum((1, 2, 3))).round().long().sub(PB // 2 - 1).clamp(1, SIZE - PB - 1)
    cx = ((m[:, 0] * xs).sum((1, 2)) / m.sum((1, 2, 3))).round().long().sub(PB // 2 - 1).clamp(1, SIZE - PB - 1)
    region = torch.zeros(n, 1, SIZE, SIZE)                        # 修复区域：以掩膜质心为中心的 8x8 方框，1 表示要重新生成
    for i in range(n):
        region[i, 0, cy[i]:cy[i] + PB, cx[i]:cx[i] + PB] = 1
    tgt_d, region_d = tgt.to(device), region.to(device)

    def replace(x, abar, g):                                      # 每一步之后：方框外换成“良品图加噪到当前噪声水平”
        return mod.inpaint_replace(x, tgt_d, region_d, abar, torch.randn(x.shape, generator=g).to(device))

    out = sample_img(base, n, seed=7, ctrl=ctrl, mask=m, after=replace)
    free = sample_img(base, n, seed=7, ctrl=ctrl, mask=m)         # 对照：采样过程中完全不管良品图
    late = region * free + (1 - region) * tgt                     # 只在最后把方框外贴回良品图

    def seam(img):
        """方框最外一圈和它外面的邻居之间的平均绝对差，对 n 张图取平均。"""
        vals = []
        for i in range(n):
            y0, x0, y1, x1 = int(cy[i]), int(cx[i]), int(cy[i]) + PB - 1, int(cx[i]) + PB - 1
            a = img[i, 0]
            vals.append(torch.cat([(a[y0, x0:x1 + 1] - a[y0 - 1, x0:x1 + 1]).abs(), (a[y1, x0:x1 + 1] - a[y1 + 1, x0:x1 + 1]).abs(),
                                   (a[y0:y1 + 1, x0] - a[y0:y1 + 1, x0 - 1]).abs(), (a[y0:y1 + 1, x1] - a[y0:y1 + 1, x1 + 1]).abs()]).mean())
        return float(torch.stack(vals).mean())

    ref = seam(good[n:2 * n])                                     # 另外 200 张没动过的良品图在同样位置上的值
    for name, x in (("每一步都替换（修复）", out), ("只在最后贴回良品图", late)):
        print(f"{name}：方框外的最大改动 {float(((x - tgt).abs() * (1 - region)).max()):.1e}，"
              f"方框内有缺陷的比例 {float(((residual(x).abs() * region).amax(dim=(1, 2, 3)) > THR).float().mean()):.2f}，"
              f"跨边界差是没动过的良品图的 {seam(x) / ref:.2f} 倍")
    fig, ax = plt.subplots(3, 8, figsize=(10, 4))
    show_row(ax[0], tgt, region, "良品图和修复区域", "r"); show_row(ax[1], out, m, "每一步都替换"); show_row(ax[2], late, m, "只在最后贴回")
    plt.tight_layout(); plt.show()

## 选做：在自己的机器上跑一次 Stable Diffusion

需要联网，要另外装包。这个单元默认跳过，写课程的机器上没有运行过，接口以 diffusers 的官方文档为准。模型的许可证各不相同，用之前先读模型卡。

In [ ]:
RUN_REAL = False          # 改成 True 再运行这个单元；要先 pip install diffusers transformers accelerate，第一次运行会联网下载几个 GB 的权重
MODEL_ID = ""             # 填一个 Stable Diffusion 1.x 权重在 Hugging Face 上的名字。先读它的模型卡，确认许可证允许你的用途
if not RUN_REAL or not MODEL_ID:
    print("跳过。把 RUN_REAL 改成 True、填上 MODEL_ID 再运行；做法、显存不够时怎么办、应该看到什么，见教程第 8 节。")
else:
    try:
        from diffusers import StableDiffusionPipeline
        half = device.type == "cuda"
        pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if half else torch.float32)
        pipe.enable_attention_slicing()                       # 注意力分片算，省显存
        pipe = pipe.to(device)
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        g = torch.Generator(device="cpu").manual_seed(0)
        image = pipe("a macro photo of a scratched metal surface", num_inference_steps=20, guidance_scale=7.5, height=512, width=512, generator=g).images[0]
        if device.type == "cuda":
            print("显存峰值（MiB）：", round(torch.cuda.max_memory_allocated() / 2 ** 20))
        plt.imshow(image); plt.axis("off"); plt.show()
    except Exception as exc:
        print("没有跑通：", type(exc).__name__, exc)

## 破坏实验与记录

- 实验 1 里把 `sample_2d` 的步数改成 2，再把 `eta` 改成 0.5，看距离和团内标准差比怎么变。
- 实验 3 里把 `train_2d` 的 `p_drop` 改成 0 重新训练一个网络，用 w = 1 和 w = 3 各采一次。先预测哪一个会坏掉。
- 实验 4 里把 `Control(base)` 改成 `Control(base, zero_init=False)`，看“训练之前输出改变的最大值”和训练后的两个数。
- 实验 5 里把秩从 4 改成 1 和 32，把 4 张图的步数从 800 改成 200 和 2000，看复制比例。
- 实验 6 里把修复区域从 8×8 的方框改成缺陷掩膜本身（`region = m.clone()`），看方框内有缺陷的比例和接缝。
- 把采样步数、引导强度、掩膜对得上的程度、复制比例这几组数填进 `实验记录.md`。